# Parte D · Nuevo trabajo: ¿predice el modelo recetas que nunca vio?

**Gap (Parte C):** el paper separa entrenamiento y prueba **por medición**, aunque cada receta se midió 3 veces. Entre el 83 y el 100 % de las filas de prueba tienen una réplica de su misma receta en entrenamiento, y un "memorizador" que no aprende nada obtiene casi el mismo r² que el Random Forest.

**Qué hace este notebook:** repite el Random Forest de la Parte B con dos formas de separar los datos (**por medición**, como el paper, y **por receta**), 100 veces cada una. Con eso contrasta las tres hipótesis formuladas en la Parte C:

| Hipótesis | Pregunta | Sección |
|---|---|---|
| **H1** | ¿Baja el r² cuando la receta de prueba es nueva? | §3 |
| **H2** | ¿El error queda cerca del ruido entre réplicas o muy por encima? | §4 |
| **H3** | ¿Los intervalos de incertidumbre tienen la cobertura que prometen? | §5 |

Además incluye:
- un control de hiperparámetros (§6);
- una prueba de sensibilidad de la EE (§7);
- el veredicto de cada hipótesis, calculado con criterios fijados de antemano (§8).

> **Diseño registrado antes de ejecutar:** `docs/parte_D_diseno.md` (17-nov-2026). Ahí están las hipótesis, los criterios de aceptación y las decisiones de diseño. Este notebook los aplica sin modificarlos.
>
> **Línea base:** la reproducción de la Parte B (`resultados/parte_B/`). Se usan el mismo preprocesamiento y los mismos hiperparámetros del RF.
>
> **Tiempo:** ~50 min con `N_JOBS = 6` (12 núcleos, 16 GB de RAM).

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import stats
from sklearn.model_selection import KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = ROOT / "datos_procesados"
PARTE_B = ROOT / "resultados" / "parte_B"
PARTE_C = ROOT / "resultados" / "parte_C"
REF_R = ROOT / "resultados" / "referencia_R"
SALIDA = ROOT / "resultados" / "parte_D"
SALIDA.mkdir(parents=True, exist_ok=True)

# Constantes del autor (iguales a las Partes B y C)
SEED = 1987          # semilla de los modelos y de los folds internos
PDI_LIM = 0.250      # filtro de los modelos de dH
TRAIN_PERC = 0.85    # proporción de entrenamiento

# Constantes de la Parte D (docs/parte_D_diseno.md)
N_REP = 100          # repeticiones de cada esquema de partición (semillas 0, 1, ..., N_REP - 1)
N_REP_ANIDADO = 20   # repeticiones del control con ajuste anidado de hiperparámetros (§6)
N_ARBOLES = 500      # árboles del RF (igual que la Parte B)
K_INTERNO = 5        # folds internos para calibrar los intervalos conformales
ALPHA = 0.10         # intervalos al 90 %
Z90 = stats.norm.ppf(1 - ALPHA / 2)
N_JOBS = 6           # procesos en paralelo (con 12 se agotó la RAM en la Parte B)

# Estilo de figuras: misma paleta categórica validada de las Partes B y C (slots 1-3)
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA2, REJILLA = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.edgecolor": TINTA2, "axes.labelcolor": TINTA, "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Datos, línea base y ruido entre réplicas

- **Datos y conjuntos de modelado:** son los mismos de la Parte B. En dH se filtra PdI ≤ 0.25. `meta_formulation_id` (la receta) **no es predictor**: solo sirve para agrupar las réplicas.
- **Hiperparámetros del RF:** se usan, **fijos**, los que eligió la Parte B para cada modelo (`mtry`, `min_n`, 500 árboles). Así, frente a la línea base, solo cambia la forma de separar los datos.
- **Ruido entre réplicas (SD agrupada):** √(Σ (medición − promedio de su receta)² / (n − n.º de recetas)). Se recalcula aquí y se comprueba que coincide con la Parte C.

In [ ]:
ML = {"lipidos": pd.read_csv(DATOS / "ml_lipidos.csv"), "plga": pd.read_csv(DATOS / "ml_plga.csv")}
FAMILIAS = {"lipidos": "Lípidos", "plga": "PLGA"}
OBJETIVOS = {"hdiam_nm": "dH (nm)", "pdi": "PdI", "percEE": "EE (%)"}
NO_PREDICTORES = ["payload", "hdiam_nm", "pdi", "zetaPot_mV", "percEE"]
MODELOS = [(f, o) for f in FAMILIAS for o in OBJETIVOS]
ETIQUETA = {(f, o): f"{FAMILIAS[f]} · {OBJETIVOS[o]}" for f, o in MODELOS}


def predictores(familia):
    return [c for c in ML[familia].columns if c not in NO_PREDICTORES and not c.startswith("meta_")]


def conjunto(familia, objetivo, variante="base"):
    # variante: "base" (Parte B), "con_farmaco" (solo recetas con fármaco) o "ee_recortada" (EE > 100 % → 100 %)
    d = ML[familia]
    if objetivo == "hdiam_nm":
        d = d[d["pdi"] <= PDI_LIM]
    if variante == "con_farmaco":
        d = d[d["reag_payload_ratio"] > 0]
    elif variante == "ee_recortada":
        d = d.assign(percEE=d["percEE"].clip(upper=100))
    return d


def sd_agrupada(d, objetivo):
    resid = d[objetivo] - d.groupby("meta_formulation_id")[objetivo].transform("mean")
    return float(np.sqrt((resid ** 2).sum() / (len(d) - d.meta_formulation_id.nunique())))


# Hiperparámetros elegidos por la Parte B (rejilla con 10-fold CV por medición)
TABLA2_B = pd.read_csv(PARTE_B / "tabla2_rf_python.csv")
INV_F = {v: k for k, v in FAMILIAS.items()}
INV_O = {v: k for k, v in OBJETIVOS.items()}
HIPER = {(INV_F[r.Familia], INV_O[r.Objetivo]): {"mtry": int(r.mtry), "min_n": int(r.min_n)}
         for r in TABLA2_B.itertuples()}
RUIDO = {(f, o): sd_agrupada(conjunto(f, o), o) for f, o in MODELOS}

ruido_c = pd.read_csv(PARTE_C / "evidencia3_ruido_replicas.csv").set_index("Modelo")
filas = []
for f, o in MODELOS:
    d = conjunto(f, o)
    filas.append({"Modelo": ETIQUETA[(f, o)], "filas": len(d), "recetas": d.meta_formulation_id.nunique(),
                  "mtry (Parte B)": HIPER[(f, o)]["mtry"], "min_n (Parte B)": HIPER[(f, o)]["min_n"],
                  "SD entre réplicas": RUIDO[(f, o)],
                  "coincide con Parte C": bool(np.isclose(RUIDO[(f, o)], ruido_c.loc[ETIQUETA[(f, o)], "SD entre réplicas (agrupada)"]))})
pd.DataFrame(filas).round(4)

## 2. Diseño del experimento

### 2.1 Los dos esquemas de partición

| Esquema | Regla | Qué mide |
|---|---|---|
| **Por medición** (paper, Parte B) | 85 % de las **filas** a entrenamiento, estratificado por cuartiles del objetivo. Es la misma función de las Partes B y C; con las semillas 0–99 se obtienen exactamente las 100 primeras particiones simuladas en la Parte C | Qué tan bien se predice **otra medición de una receta ya vista** |
| **Por receta** (propuesto) | 85 % de las **recetas** a entrenamiento, estratificado por cuartiles del **promedio de la receta**. Las 3 réplicas van siempre juntas | Qué tan bien se predice **una receta nueva**, que es el uso real del modelo (diseño inverso) |

La celda siguiente comprueba la diferencia en la semilla 0: con partición por receta, **ninguna** fila de prueba tiene su réplica en entrenamiento.

In [ ]:
def estratos_rsample(y, cortes=4, pool=0.10):
    # Emulación de rsample: cuartiles del objetivo; los estratos con < 10 % de los datos se fusionan con el vecino
    e = np.asarray(pd.qcut(y, q=cortes, labels=False, duplicates="drop"))
    while True:
        etiquetas, n = np.unique(e, return_counts=True)
        if len(etiquetas) <= 1 or n.min() >= pool * len(y):
            return e
        i = int(np.argmin(n))
        e[e == etiquetas[i]] = etiquetas[i - 1] if i > 0 else etiquetas[i + 1]


def particion_por_medicion(d, objetivo, semilla):
    # Igual que la Parte B / C: initial_split(prop = 0.85, strata = objetivo) sobre FILAS
    y = d[objetivo].to_numpy()
    rng = np.random.default_rng(semilla)
    estratos = estratos_rsample(y)
    en_train = np.zeros(len(y), dtype=bool)
    for e in np.unique(estratos):
        idx = np.flatnonzero(estratos == e)
        en_train[rng.choice(idx, int(np.floor(len(idx) * TRAIN_PERC)), replace=False)] = True
    return en_train


def particion_por_receta(d, objetivo, semilla):
    # La misma regla, pero sobre RECETAS: se estratifica por el promedio de cada receta
    medias = d.groupby("meta_formulation_id")[objetivo].mean()
    rng = np.random.default_rng(semilla)
    estratos = estratos_rsample(medias.to_numpy())
    recetas_train = []
    for e in np.unique(estratos):
        ids = medias.index[estratos == e].to_numpy()
        recetas_train.extend(rng.choice(ids, int(np.floor(len(ids) * TRAIN_PERC)), replace=False))
    return d["meta_formulation_id"].isin(recetas_train).to_numpy()


filas = []
for f, o in MODELOS:
    d = conjunto(f, o)
    g = d.meta_formulation_id.to_numpy()
    for nombre, regla in [("por medición", particion_por_medicion), ("por receta", particion_por_receta)]:
        tr = regla(d, o, 0)
        filas.append({"Modelo": ETIQUETA[(f, o)], "esquema": nombre, "n train": int(tr.sum()), "n test": int((~tr).sum()),
                      "% filas test con réplica en train": 100 * np.isin(g[~tr], g[tr]).mean()})
pd.DataFrame(filas).pivot(index="Modelo", columns="esquema").round(1)

### 2.2 Modelos, métricas e intervalos

**Modelos.** En cada repetición y esquema se ajustan tres:
- **Random Forest:** la misma tubería de la Parte B (`VarianceThreshold` → `StandardScaler` → RF), con 500 árboles y los `mtry` y `min_n` de la Parte B.
- **Regresión lineal:** la misma tubería. Sirve de referencia simple.
- **Memorizador:** predice con el promedio de las réplicas de la receta que están en entrenamiento; si no hay ninguna, con la media global. Con partición por receta se reduce siempre a la media global, así que su r² (correlación²) no está definido.

**Métricas.**
- **r² (correlación²):** la definición del autor. Es la principal, para comparar con la Tabla 2.
- **r² = 1 − SSE/SST:** penaliza el sesgo y puede ser negativo.
- **RMSE.**

**Intervalos al 90 %** (solo con partición por receta, H3):

| Método | Construcción |
|---|---|
| **Conformal calibrado por receta** (principal) | 1. Validación cruzada por grupos (`GroupKFold`, 5 folds) **dentro del entrenamiento**. 2. Se toman los residuos absolutos fuera de fold y su cuantil q̂ de nivel ⌈(n+1)·0.9⌉/n. 3. El intervalo es la predicción del RF entrenado con todo ± q̂ |
| **Conformal calibrado por medición** | Igual, pero con `KFold` sobre **filas**, que es la validación del paper. Permite ver si calibrar con fuga da intervalos demasiado optimistas |
| **Quantile Random Forest** (Meinshausen, 2006) | Implementación propia sobre el **mismo** RF. Para cada fila de prueba, cada fila de entrenamiento recibe un peso proporcional a cuántas veces comparte hoja con ella. El intervalo son los cuantiles ponderados 5 % y 95 % de y |

**Supuesto:** el conformal por validación cruzada usa el modelo entrenado con todo el entrenamiento y los residuos fuera de fold. Es una versión simplificada de CV+ (Barber *et al.*, 2021). Su cobertura es aproximada, no exacta, porque los modelos de cada fold se entrenan con un poco menos de datos.

In [ ]:
def tuberia(modelo):
    return Pipeline([("zv", VarianceThreshold(0.0)), ("esc", StandardScaler()), ("modelo", modelo)])


def nuevo_rf(h, X_tr, n_arboles=N_ARBOLES):
    # Igual que fabrica_rf de la Parte B. mtry se limita a las columnas con varianza > 0 del entrenamiento
    p = int((np.var(X_tr, axis=0) > 0).sum())
    return tuberia(RandomForestRegressor(n_estimators=n_arboles, max_features=min(h["mtry"], p),
                                         min_samples_split=h["min_n"] + 1, random_state=SEED, n_jobs=1))


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


def rsq_cor(y, p):
    # r² del autor: correlación de Pearson al cuadrado (NaN si la predicción es constante)
    y, p = np.asarray(y, float), np.asarray(p, float)
    if np.std(p) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return float(np.corrcoef(y, p)[0, 1] ** 2)


def metricas(prefijo, y, p):
    return {f"{prefijo}_rmse": rmse(y, p), f"{prefijo}_rsq": rsq_cor(y, p), f"{prefijo}_r2sse": float(r2_score(y, p))}


def memorizador(y_tr, g_tr, g_te):
    medias = pd.Series(y_tr).groupby(g_tr).mean()
    return pd.Series(g_te).map(medias).fillna(float(np.mean(y_tr))).to_numpy()


def cobertura(prefijo, y, lo, hi):
    return {f"{prefijo}_cobertura": 100 * float(np.mean((y >= lo) & (y <= hi))), f"{prefijo}_ancho": float(np.mean(hi - lo))}


def residuos_fuera_de_fold(h, X, y, grupos, por_receta):
    # Residuos absolutos de validación cruzada interna: por receta (GroupKFold) o por medición (KFold)
    if por_receta:
        cv = GroupKFold(n_splits=K_INTERNO, shuffle=True, random_state=SEED).split(X, y, grupos)
    else:
        cv = KFold(n_splits=K_INTERNO, shuffle=True, random_state=SEED).split(X)
    oof = np.empty(len(y))
    for a, b in cv:
        oof[b] = nuevo_rf(h, X[a]).fit(X[a], y[a]).predict(X[b])
    return np.abs(y - oof)


def cuantil_conformal(residuos, alpha=ALPHA):
    n = len(residuos)
    nivel = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(residuos, nivel, method="higher"))


def intervalo_qrf(modelo, X_tr, y_tr, X_te, alpha=ALPHA):
    # Quantile Random Forest (Meinshausen 2006) sobre el RF ya entrenado
    prep, rf = modelo[:-1], modelo[-1]
    hojas_tr = rf.apply(prep.transform(X_tr))
    hojas_te = rf.apply(prep.transform(X_te))
    pesos = np.zeros((len(X_te), len(X_tr)))
    for t in range(hojas_tr.shape[1]):
        misma_hoja = hojas_te[:, t][:, None] == hojas_tr[:, t][None, :]
        pesos += misma_hoja / misma_hoja.sum(axis=1, keepdims=True)
    pesos /= hojas_tr.shape[1]
    orden = np.argsort(y_tr)
    acumulado = np.cumsum(pesos[:, orden], axis=1)
    y_ordenada = y_tr[orden]

    def cuantil(q):
        return y_ordenada[(acumulado >= q - 1e-12).argmax(axis=1)]

    return cuantil(alpha / 2), cuantil(1 - alpha / 2)

### 2.3 Una repetición

La función `una_repeticion(familia, objetivo, semilla)` hace lo siguiente:

1. **Para cada esquema** (por medición y por receta): separa los datos, ajusta RF, regresión lineal y memorizador, y calcula sus métricas en la prueba.
2. **Solo con partición por receta:**
   - **Intervalos (H3):** construye los tres intervalos al 90 % y mide su cobertura y su ancho.
   - **Experimento controlado de fuga (H1-control):** de cada receta de prueba se toma **una réplica al azar** como fila de prueba y se comparan dos RF:
     - **sin fuga:** el RF de esta misma repetición;
     - **con fuga:** un RF que además vio las otras réplicas de esas recetas.

     Como las filas de prueba son idénticas, la diferencia se debe **solo** a haber visto las réplicas.

In [ ]:
def una_repeticion(familia, objetivo, semilla, variante="base", extras=True):
    d = conjunto(familia, objetivo, variante)
    X = d[predictores(familia)].to_numpy(float)
    y = d[objetivo].to_numpy(float)
    g = d["meta_formulation_id"].to_numpy()
    h = HIPER[(familia, objetivo)]
    filas = []
    for esquema, regla in [("medicion", particion_por_medicion), ("receta", particion_por_receta)]:
        tr = regla(d, objetivo, semilla)
        te = ~tr
        rf = nuevo_rf(h, X[tr]).fit(X[tr], y[tr])
        p_rf = rf.predict(X[te])
        fila = {"familia": familia, "objetivo": objetivo, "variante": variante, "semilla": semilla, "esquema": esquema,
                "n_train": int(tr.sum()), "n_test": int(te.sum()), "recetas_test": len(np.unique(g[te])),
                "pct_test_con_replica": 100 * float(np.isin(g[te], g[tr]).mean())}
        fila.update(metricas("rf", y[te], p_rf))
        lineal = tuberia(LinearRegression()).fit(X[tr], y[tr])
        fila.update(metricas("lineal", y[te], lineal.predict(X[te])))
        fila.update(metricas("memo", y[te], memorizador(y[tr], g[tr], g[te])))

        if extras and esquema == "receta":
            # H3: intervalos al 90 %
            for nombre, por_receta in [("conf_receta", True), ("conf_medicion", False)]:
                q = cuantil_conformal(residuos_fuera_de_fold(h, X[tr], y[tr], g[tr], por_receta))
                fila.update(cobertura(nombre, y[te], p_rf - q, p_rf + q))
            lo, hi = intervalo_qrf(rf, X[tr], y[tr], X[te])
            fila.update(cobertura("qrf", y[te], lo, hi))

            # H1-control: mismas filas de prueba, con y sin las réplicas hermanas en el entrenamiento
            rng = np.random.default_rng(10_000 + semilla)
            idx_prueba = np.array([rng.choice(np.flatnonzero(g == r)) for r in np.unique(g[te])])
            hermanas = np.setdiff1d(np.flatnonzero(te), idx_prueba)
            idx_fuga = np.r_[np.flatnonzero(tr), hermanas]
            rf_fuga = nuevo_rf(h, X[idx_fuga]).fit(X[idx_fuga], y[idx_fuga])
            fila.update({"ctrl_n_prueba": len(idx_prueba),
                         "ctrl_pct_con_hermana": 100 * float(np.isin(g[idx_prueba], g[hermanas]).mean())})
            fila.update(metricas("ctrl_sin_fuga", y[idx_prueba], rf.predict(X[idx_prueba])))
            fila.update(metricas("ctrl_con_fuga", y[idx_prueba], rf_fuga.predict(X[idx_prueba])))
        filas.append(fila)
    return filas

In [ ]:
# Experimento principal: 6 modelos × N_REP repeticiones × 2 esquemas (~35 min con 6 procesos)
t0 = time.time()
salidas = Parallel(n_jobs=N_JOBS)(delayed(una_repeticion)(f, o, s) for f, o in MODELOS for s in range(N_REP))
REP = pd.DataFrame([fila for filas in salidas for fila in filas])
REP.to_csv(SALIDA / "repeticiones.csv", index=False)
print(f"{len(REP)} filas ({N_REP} repeticiones × 6 modelos × 2 esquemas) en {(time.time() - t0) / 60:.1f} min")
REP.groupby(["familia", "objetivo", "esquema"])[["n_train", "n_test", "recetas_test", "pct_test_con_replica"]].mean().round(1)

## 3. H1 · ¿Baja el r² cuando la receta de prueba es nueva?

> **H1:** con partición por receta, el r² de prueba del RF **baja** frente a la partición por medición, sobre todo en **PdI** y en **dH de lípidos**. El RMSE de prueba sube, como mínimo, hasta superar el ruido entre réplicas.

### 3.1 Resultados de las 100 repeticiones

Cada celda de la tabla da la **mediana [percentil 5 – percentil 95]** de las 100 repeticiones.

In [ ]:
def mediana_rango(s, dec=3):
    s = pd.Series(s).dropna()
    if s.empty:
        return "—"
    return f"{s.median():.{dec}f} [{s.quantile(0.05):.{dec}f} – {s.quantile(0.95):.{dec}f}]"


def dec(objetivo):
    return 4 if objetivo == "pdi" else 2


filas = []
for f, o in MODELOS:
    r = REP[(REP.familia == f) & (REP.objetivo == o)]
    m, g = r[r.esquema == "medicion"], r[r.esquema == "receta"]
    filas.append({
        "Modelo": ETIQUETA[(f, o)],
        "r² RF · por medición": mediana_rango(m.rf_rsq), "r² RF · por receta": mediana_rango(g.rf_rsq),
        "RMSE RF · por medición": mediana_rango(m.rf_rmse, dec(o)), "RMSE RF · por receta": mediana_rango(g.rf_rmse, dec(o)),
        "r² (1−SSE/SST) RF · por receta": mediana_rango(g.rf_r2sse),
        "r² lineal · por receta": mediana_rango(g.lineal_rsq),
        "r² memorizador · por medición": mediana_rango(m.memo_rsq),
    })
H1_TABLA = pd.DataFrame(filas)
H1_TABLA.to_csv(SALIDA / "h1_resumen.csv", index=False)
H1_TABLA

### 3.2 Pruebas estadísticas (criterio H1a, fijado de antemano)

- **Diferencia por repetición:** d = r²(por medición) − r²(por receta), emparejando las repeticiones por su número de semilla.
- **Prueba *t* corregida de Nadeau y Bengio (2003):** t = media(d) / √((1/J + n_prueba/n_entren.) · var(d)), con J − 1 grados de libertad. El término n_prueba/n_entren. corrige que los entrenamientos de las distintas repeticiones se solapan. Es conservadora.
- **Wilcoxon de rangos con signo:** prueba no paramétrica, como respaldo.
- **Corrección de Holm** para los 6 modelos.
- **Criterio:** H1a se acepta en un modelo si la mediana de d es > 0 **y** el p de Holm de la *t* corregida es < 0.05.

In [ ]:
def t_corregida(d, ratio_prueba_entren):
    d = np.asarray(d, float)
    d = d[~np.isnan(d)]
    J = len(d)
    t = d.mean() / np.sqrt((1 / J + ratio_prueba_entren) * d.var(ddof=1))
    return float(t), float(2 * stats.t.sf(abs(t), J - 1))


def holm(p):
    p = np.asarray(p, float)
    orden = np.argsort(p)
    ajustado, acumulado = np.empty(len(p)), 0.0
    for k, i in enumerate(orden):
        acumulado = max(acumulado, (len(p) - k) * p[i])
        ajustado[i] = min(1.0, acumulado)
    return ajustado


def pareado(f, o, columna):
    r = REP[(REP.familia == f) & (REP.objetivo == o)]
    a = r[r.esquema == "medicion"].set_index("semilla")[columna]
    b = r[r.esquema == "receta"].set_index("semilla")[columna]
    return (a - b).dropna()


filas = []
for f, o in MODELOS:
    r = REP[(REP.familia == f) & (REP.objetivo == o)]
    ratio = float((r.n_test / r.n_train).mean())
    d = pareado(f, o, "rf_rsq")
    t, p = t_corregida(d, ratio)
    filas.append({"Modelo": ETIQUETA[(f, o)], "familia": f, "objetivo": o, "repeticiones": len(d),
                  "mediana caída r²": d.median(), "% repeticiones con caída": 100 * (d > 0).mean(),
                  "t corregida": t, "p (t corregida)": p, "p (Wilcoxon)": float(stats.wilcoxon(d).pvalue),
                  "mediana aumento RMSE": -pareado(f, o, "rf_rmse").median()})
H1_PRUEBAS = pd.DataFrame(filas)
H1_PRUEBAS["p Holm (t corregida)"] = holm(H1_PRUEBAS["p (t corregida)"])
H1_PRUEBAS["p Holm (Wilcoxon)"] = holm(H1_PRUEBAS["p (Wilcoxon)"])
H1_PRUEBAS["H1a"] = np.where((H1_PRUEBAS["mediana caída r²"] > 0) & (H1_PRUEBAS["p Holm (t corregida)"] < 0.05),
                             "✅ se acepta", "❌ se rechaza")
H1_PRUEBAS.to_csv(SALIDA / "h1_pruebas.csv", index=False)
H1_PRUEBAS.drop(columns=["familia", "objetivo"]).round(4)

In [ ]:
# Figura H1: distribución del r² del RF en las 100 repeticiones de cada esquema
TABLA2_PAPER = {("lipidos", "hdiam_nm"): 0.922, ("lipidos", "pdi"): 0.787, ("lipidos", "percEE"): 0.472,
                ("plga", "hdiam_nm"): 0.964, ("plga", "pdi"): 0.560, ("plga", "percEE"): 0.984}
rng_fig = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(11, 4.8))
for i, (f, o) in enumerate(MODELOS):
    r = REP[(REP.familia == f) & (REP.objetivo == o)]
    for k, (esq, color) in enumerate([("medicion", C1), ("receta", C2)]):
        v = r[r.esquema == esq].rf_rsq.dropna().to_numpy()
        x = i + (k - 0.5) * 0.36
        ax.scatter(x + rng_fig.uniform(-0.09, 0.09, len(v)), v, s=10, color=color, alpha=0.35, linewidth=0, zorder=2)
        ax.plot([x - 0.13, x + 0.13], [np.median(v)] * 2, color=color, lw=2.5, zorder=3, solid_capstyle="round")
    ax.scatter(i - 0.18, TABLA2_PAPER[(f, o)], marker="D", s=46, color=TINTA, edgecolor="white", linewidth=1.2, zorder=4)
ax.scatter([], [], s=30, color=C1, label="partición por medición (como el paper)")
ax.scatter([], [], s=30, color=C2, label="partición por receta (recetas nuevas)")
ax.plot([], [], color=TINTA2, lw=2.5, label="mediana de 100 repeticiones")
ax.scatter([], [], marker="D", s=40, color=TINTA, label="Tabla 2 del paper")
ax.set_xticks(range(len(MODELOS)), [ETIQUETA[m] for m in MODELOS], rotation=15, ha="right")
ax.set_ylabel("r² de prueba del Random Forest\n(correlación², definición del autor)")
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=4)
ax.set_title("H1 · Con recetas nuevas en la prueba, el r² del Random Forest cae y se vuelve más incierto", color=TINTA)
fig.tight_layout()
fig.savefig(SALIDA / "fig_h1_r2_por_esquema.png", bbox_inches="tight")
plt.show()

### 3.3 Subproducto: la Tabla 2 con 100 semillas

Las 100 repeticiones **por medición** son exactamente el procedimiento del paper, repetido con 100 semillas. Permiten ubicar en esa distribución el valor publicado, el del código R del autor y el de la Parte B. Con eso se confirma, ahora con el **RF real**, lo que en `docs/reproduccion_paper.md` §5.3 se aproximó con el memorizador.

El **percentil** es el % de repeticiones con un r² menor o igual que el valor indicado.

In [ ]:
def percentil_de(valor, muestra):
    muestra = np.asarray(muestra, float)
    muestra = muestra[~np.isnan(muestra)]
    return 100 * float(np.mean(muestra <= valor))


t2_r = pd.read_csv(REF_R / "tabla2_rf_R.csv").set_index(["familia", "objetivo"]) if (REF_R / "tabla2_rf_R.csv").exists() else None
filas = []
for f, o in MODELOS:
    v = REP[(REP.familia == f) & (REP.objetivo == o) & (REP.esquema == "medicion")].rf_rsq
    py_b = float(TABLA2_B[(TABLA2_B.Familia == FAMILIAS[f]) & (TABLA2_B.Objetivo == OBJETIVOS[o])].rsq_test.iloc[0])
    fila = {"Modelo": ETIQUETA[(f, o)], "r² RF por medición: mediana [p5 – p95]": mediana_rango(v),
            "paper": TABLA2_PAPER[(f, o)], "percentil paper": percentil_de(TABLA2_PAPER[(f, o)], v),
            "Python Parte B": py_b, "percentil Parte B": percentil_de(py_b, v)}
    if t2_r is not None:
        fila["código R del autor"] = float(t2_r.loc[(f, o), "rsq_test"])
        fila["percentil código R"] = percentil_de(fila["código R del autor"], v)
    filas.append(fila)
TABLA2_SEMILLAS = pd.DataFrame(filas)
TABLA2_SEMILLAS.to_csv(SALIDA / "tabla2_100_semillas.csv", index=False)
TABLA2_SEMILLAS.round(3)

### 3.4 Experimento controlado: la fuga aislada

La comparación anterior usa filas de prueba distintas en cada esquema. Aquí las filas de prueba son **las mismas**: una réplica por receta de prueba. Se comparan dos RF:
- **sin fuga:** no vio ninguna réplica de esas recetas;
- **con fuga:** vio las otras réplicas, como pasa en la partición del paper.

**Criterio (H1-control):** la mediana de la diferencia RMSE(sin fuga) − RMSE(con fuga) es > 0, y el p de Holm (Wilcoxon pareado) es < 0.05.

En los modelos de dH, el filtro PdI ≤ 0.25 deja algunas recetas con una sola réplica. En esas recetas no hay réplicas hermanas que añadir, y la columna `% filas con réplica hermana` lo indica.

In [ ]:
rec = REP[REP.esquema == "receta"]
filas = []
for f, o in MODELOS:
    r = rec[(rec.familia == f) & (rec.objetivo == o)]
    dif = (r.ctrl_sin_fuga_rmse - r.ctrl_con_fuga_rmse).to_numpy()
    filas.append({"Modelo": ETIQUETA[(f, o)], "filas de prueba por repetición": r.ctrl_n_prueba.mean(),
                  "% filas con réplica hermana": r.ctrl_pct_con_hermana.mean(),
                  "RMSE con fuga (mediana)": r.ctrl_con_fuga_rmse.median(),
                  "RMSE sin fuga (mediana)": r.ctrl_sin_fuga_rmse.median(),
                  "cociente sin / con fuga (mediana)": (r.ctrl_sin_fuga_rmse / r.ctrl_con_fuga_rmse).median(),
                  "r² con fuga (mediana)": r.ctrl_con_fuga_rsq.median(), "r² sin fuga (mediana)": r.ctrl_sin_fuga_rsq.median(),
                  "mediana diferencia RMSE": float(np.median(dif)), "p (Wilcoxon)": float(stats.wilcoxon(dif).pvalue)})
CONTROL_FUGA = pd.DataFrame(filas)
CONTROL_FUGA["p Holm"] = holm(CONTROL_FUGA["p (Wilcoxon)"])
CONTROL_FUGA["H1-control"] = np.where((CONTROL_FUGA["mediana diferencia RMSE"] > 0) & (CONTROL_FUGA["p Holm"] < 0.05),
                                      "✅ se acepta", "❌ se rechaza")
CONTROL_FUGA.to_csv(SALIDA / "h1_control_fuga.csv", index=False)
CONTROL_FUGA.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
rng_fig = np.random.default_rng(SEED)
for i, (f, o) in enumerate(MODELOS):
    r = rec[(rec.familia == f) & (rec.objetivo == o)]
    v = (r.ctrl_sin_fuga_rmse / r.ctrl_con_fuga_rmse).to_numpy()
    ax.scatter(i + rng_fig.uniform(-0.15, 0.15, len(v)), v, s=12, color=C2, alpha=0.4, linewidth=0, zorder=2)
    ax.plot([i - 0.22, i + 0.22], [np.median(v)] * 2, color=TINTA, lw=2.5, zorder=3, solid_capstyle="round")
ax.axhline(1, color=TINTA2, lw=1, ls="--", zorder=1)
ax.text(1.005, 1, "sin diferencia", transform=ax.get_yaxis_transform(), color=TINTA2, fontsize=9, ha="left", va="center")
ax.set_xticks(range(len(MODELOS)), [ETIQUETA[m] for m in MODELOS], rotation=15, ha="right")
ax.set_ylabel("RMSE sin fuga / RMSE con fuga\n(mismas filas de prueba)")
ax.set_title("Experimento controlado · Cuánto empeora el error solo por no haber visto las réplicas", color=TINTA)
ax.scatter([], [], s=30, color=C2, label="una repetición")
ax.plot([], [], color=TINTA, lw=2.5, label="mediana")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2)
fig.tight_layout()
fig.savefig(SALIDA / "fig_h1_control_fuga.png", bbox_inches="tight")
plt.show()

## 4. H2 · ¿El error queda cerca del ruido entre réplicas?

> **H2:** con la partición por receta, el error del RF en **dH y EE de PLGA** se mantiene en el orden del ruido entre réplicas; en **PdI** queda claramente por encima.

**Cociente = RMSE de prueba / SD entre réplicas.** Un cociente de 1 significa que el modelo se equivoca tanto como difieren entre sí dos mediciones de la misma receta, que es el límite físico.

**Criterio:** mediana del cociente (partición por receta) **≤ 1.5** en PLGA · dH y PLGA · EE, y **> 2** en los dos modelos de PdI.

In [ ]:
REP["cociente_ruido"] = REP.rf_rmse / [RUIDO[(f, o)] for f, o in zip(REP.familia, REP.objetivo)]
filas = []
for f, o in MODELOS:
    r = REP[(REP.familia == f) & (REP.objetivo == o)]
    filas.append({"Modelo": ETIQUETA[(f, o)], "familia": f, "objetivo": o, "SD entre réplicas": RUIDO[(f, o)],
                  "RMSE / SD · por medición": r[r.esquema == "medicion"].cociente_ruido.median(),
                  "RMSE / SD · por receta": r[r.esquema == "receta"].cociente_ruido.median(),
                  "p5 – p95 (por receta)": mediana_rango(r[r.esquema == "receta"].cociente_ruido, 2).split(" ", 1)[1]})
H2_TABLA = pd.DataFrame(filas)
H2_TABLA.to_csv(SALIDA / "h2_error_vs_ruido.csv", index=False)
H2_TABLA.drop(columns=["familia", "objetivo"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(MODELOS))
for k, (esq, color, nombre) in enumerate([("medicion", C1, "partición por medición"), ("receta", C2, "partición por receta")]):
    for i, (f, o) in enumerate(MODELOS):
        v = REP[(REP.familia == f) & (REP.objetivo == o) & (REP.esquema == esq)].cociente_ruido
        xi = i + (k - 0.5) * 0.3
        ax.plot([xi, xi], [v.quantile(0.05), v.quantile(0.95)], color=color, lw=2, alpha=0.55, solid_capstyle="round")
        ax.scatter(xi, v.median(), s=60, color=color, edgecolor="white", linewidth=1.5, zorder=3,
                   label=nombre if i == 0 else None)
for nivel, texto in [(1, "= ruido entre réplicas"), (1.5, "criterio «cerca del ruido» (1.5)"), (2, "criterio «muy por encima» (2)")]:
    ax.axhline(nivel, color=TINTA2, lw=1, ls="--" if nivel == 1 else ":", zorder=1)
    ax.text(1.005, nivel, texto, transform=ax.get_yaxis_transform(), color=TINTA2, fontsize=8.5, va="center", ha="left")
ax.set_xticks(x, [ETIQUETA[m] for m in MODELOS], rotation=15, ha="right")
ax.set_ylabel("RMSE de prueba / SD entre réplicas")
ax.set_ylim(0, None)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2, title="punto = mediana · barra = percentiles 5–95",
          title_fontsize=9)
ax.set_title("H2 · Error del Random Forest medido en unidades de ruido experimental", color=TINTA)
fig.tight_layout()
fig.savefig(SALIDA / "fig_h2_error_vs_ruido.png", bbox_inches="tight")
plt.show()

## 5. H3 · ¿Los intervalos tienen la cobertura que prometen?

> **H3:** los intervalos de incertidumbre, evaluados con partición por receta, tienen **cobertura cercana a la nominal** en dH de PLGA y son **más anchos** (en relación con el ruido) en PdI y en lípidos.

Todos los intervalos son al **90 %**. Se evalúan sobre las mediciones de prueba de **recetas nuevas**.

**Criterios:**
- **H3a:** la cobertura media del conformal calibrado por receta en PLGA · dH está entre 85 y 95 %.
- **H3b:** su ancho relativo al ruido es mayor en los 2 modelos de PdI y en los 3 de lípidos que en PLGA · dH.
- **H3c** (expectativa secundaria): el conformal calibrado **por medición** cubre menos del 85 % en al menos 3 de los 6 modelos.

**Ancho relativo al ruido** = ancho medio / (2 · 1.645 · SD entre réplicas). Es el ancho del intervalo dividido por el de un intervalo al 90 % que solo tuviera en cuenta el ruido experimental.

In [ ]:
METODOS = [("conf_receta", "conformal calibrado por receta", C1), ("conf_medicion", "conformal calibrado por medición", C2),
           ("qrf", "Quantile Random Forest", C3)]
filas = []
for f, o in MODELOS:
    r = rec[(rec.familia == f) & (rec.objetivo == o)]
    fila = {"Modelo": ETIQUETA[(f, o)], "familia": f, "objetivo": o}
    for m, nombre, _ in METODOS:
        fila[f"cobertura % · {nombre}"] = r[f"{m}_cobertura"].mean()
        fila[f"ancho relativo · {nombre}"] = (r[f"{m}_ancho"] / (2 * Z90 * RUIDO[(f, o)])).median()
    fila["ancho medio · conformal por receta (unidades del objetivo)"] = r["conf_receta_ancho"].median()
    filas.append(fila)
H3_TABLA = pd.DataFrame(filas)
H3_TABLA.to_csv(SALIDA / "h3_intervalos.csv", index=False)
H3_TABLA.drop(columns=["familia", "objetivo"]).round(2).T

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5.2))
x = np.arange(len(MODELOS))
for k, (m, nombre, color) in enumerate(METODOS):
    xi = x + (k - 1) * 0.24
    ejes[0].scatter(xi, H3_TABLA[f"cobertura % · {nombre}"], s=60, color=color, edgecolor="white", linewidth=1.5,
                    zorder=3, label=nombre)
    ejes[1].scatter(xi, H3_TABLA[f"ancho relativo · {nombre}"], s=60, color=color, edgecolor="white", linewidth=1.5,
                    zorder=3, label=nombre)
ejes[0].axhspan(85, 95, color=REJILLA, alpha=0.7, zorder=0)
ejes[0].axhline(90, color=TINTA2, lw=1, ls="--", zorder=1)
ejes[0].text(-0.4, 95.5, "zona aceptada (85–95 %)", color=TINTA2, fontsize=8.5, va="bottom")
ejes[0].set_ylabel("Cobertura media (%) en recetas nuevas")
ejes[0].set_title("Cobertura de los intervalos al 90 %", color=TINTA)
ejes[1].axhline(1, color=TINTA2, lw=1, ls="--", zorder=1)
ejes[1].text(-0.45, 1.0, "= solo ruido entre réplicas", color=TINTA2, fontsize=8.5, ha="left", va="bottom")
ejes[1].set_ylabel("Ancho del intervalo / ancho del ruido")
ejes[1].set_ylim(0, None)
ejes[1].set_title("Ancho relativo al ruido experimental (mediana)", color=TINTA)
for ax in ejes:
    ax.set_xticks(x, [ETIQUETA[mo] for mo in MODELOS], rotation=25, ha="right")
fig.legend(*ejes[0].get_legend_handles_labels(), loc="lower center", ncol=3)
fig.suptitle("H3 · Incertidumbre de las predicciones para recetas nuevas", color=TINTA, fontsize=12)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(SALIDA / "fig_h3_intervalos.png", bbox_inches="tight")
plt.show()

## 6. Control: ¿la caída se debe a los hiperparámetros fijos?

Los hiperparámetros de la Parte B se eligieron con validación **por medición**. Eso puede favorecer modelos que memorizan, como `min_n = 2`, y explicar parte de la caída de H1.

**Control:** en **PLGA · PdI**, el modelo con más recetas y el más difícil, se repiten las primeras 20 particiones por receta. En cada una se ajustan los hiperparámetros con validación cruzada anidada **por grupos**:
- `GroupKFold` de 5 folds dentro del entrenamiento;
- `mtry` de 1 a p;
- `min_n` ∈ {2, 5, 10, 20, 40};
- 300 árboles durante la búsqueda y 500 en el modelo final.

**Criterio:** si la mejora mediana de r² del RF ajustado frente al RF fijo es **< 0.05**, los hiperparámetros fijos no explican el resultado de H1.

In [ ]:
F_C, O_C = "plga", "pdi"
D_C = conjunto(F_C, O_C)
X_C, Y_C, G_C = D_C[predictores(F_C)].to_numpy(float), D_C[O_C].to_numpy(float), D_C.meta_formulation_id.to_numpy()
REJILLA_C = [{"mtry": m, "min_n": n} for m in range(1, X_C.shape[1] + 1) for n in [2, 5, 10, 20, 40]]


def cv_interna(semilla, i):
    tr = particion_por_receta(D_C, O_C, semilla)
    Xt, yt, gt = X_C[tr], Y_C[tr], G_C[tr]
    errores = [rmse(yt[b], nuevo_rf(REJILLA_C[i], Xt[a], 300).fit(Xt[a], yt[a]).predict(Xt[b]))
               for a, b in GroupKFold(n_splits=K_INTERNO, shuffle=True, random_state=SEED).split(Xt, yt, gt)]
    return semilla, i, float(np.mean(errores))


def evaluar_final(semilla, i_mejor):
    tr = particion_por_receta(D_C, O_C, semilla)
    te = ~tr
    fila = {"semilla": semilla, "mtry elegido": REJILLA_C[i_mejor]["mtry"], "min_n elegido": REJILLA_C[i_mejor]["min_n"]}
    for nombre, h in [("ajustado", REJILLA_C[i_mejor]), ("fijo", HIPER[(F_C, O_C)])]:
        p = nuevo_rf(h, X_C[tr]).fit(X_C[tr], Y_C[tr]).predict(X_C[te])
        fila.update(metricas(nombre, Y_C[te], p))
    return fila


t0 = time.time()
cvi = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(cv_interna)(s, i) for s in range(N_REP_ANIDADO) for i in range(len(REJILLA_C))),
                   columns=["semilla", "config", "rmse_cv"])
mejores = cvi.loc[cvi.groupby("semilla").rmse_cv.idxmin()]
CONTROL_HIPER = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(evaluar_final)(int(r.semilla), int(r.config)) for r in mejores.itertuples()))
CONTROL_HIPER.to_csv(SALIDA / "control_hiperparametros_plga_pdi.csv", index=False)
print(f"Tiempo: {(time.time() - t0) / 60:.1f} min · {len(REJILLA_C)} configuraciones × {N_REP_ANIDADO} particiones")

mejora = CONTROL_HIPER.ajustado_rsq - CONTROL_HIPER.fijo_rsq
r_med = REP[(REP.familia == F_C) & (REP.objetivo == O_C) & (REP.esquema == "medicion")].rf_rsq
CONTROL_HIPER_RESUMEN = pd.DataFrame([{
    "r² RF fijo (por receta)": CONTROL_HIPER.fijo_rsq.median(), "r² RF ajustado (por receta)": CONTROL_HIPER.ajustado_rsq.median(),
    "r² RF fijo (por medición, 100 rep.)": r_med.median(), "mejora mediana de r² al ajustar": mejora.median(),
    "p (Wilcoxon)": float(stats.wilcoxon(mejora).pvalue),
    "min_n más elegido": int(CONTROL_HIPER["min_n elegido"].mode()[0]), "mtry más elegido": int(CONTROL_HIPER["mtry elegido"].mode()[0]),
    "conclusión": "los hiperparámetros fijos NO explican H1" if mejora.median() < 0.05 else "el ajuste por grupos mejora ≥ 0.05",
}])
CONTROL_HIPER_RESUMEN.round(4).T

## 7. Sensibilidad: ¿qué tanto del r² de la EE es solo "lleva fármaco o no"?

En los datos, **toda receta sin fármaco tiene EE = 0**. Son 162 de las 243 filas de lípidos (todas las LNP y la mitad de los liposomas) y 81 de las 324 de PLGA. Un modelo puede obtener un r² alto solo aprendiendo esa regla, sin entender la encapsulación.

Variantes (exploratorias, sin criterio de aceptación; 100 repeticiones de cada esquema, mismo RF):

| Variante | Qué cambia |
|---|---|
| **Solo recetas con fármaco** | Lípidos: 81 filas, 27 recetas (liposomas con curcumina). PLGA: 243 filas, 81 recetas |
| **EE recortada a 100 %** | Lípidos: 9 mediciones de liposomas con EE > 100 % se fijan en 100. En PLGA no hay casos |

In [ ]:
TAREAS_SENS = [("lipidos", "percEE", "con_farmaco"), ("plga", "percEE", "con_farmaco"), ("lipidos", "percEE", "ee_recortada")]
t0 = time.time()
salidas = Parallel(n_jobs=N_JOBS)(delayed(una_repeticion)(f, o, s, v, False) for f, o, v in TAREAS_SENS for s in range(N_REP))
SENS = pd.DataFrame([fila for filas in salidas for fila in filas])
SENS.to_csv(SALIDA / "sensibilidad_ee_repeticiones.csv", index=False)
print(f"Tiempo: {(time.time() - t0) / 60:.1f} min")

base_ee = REP[REP.objetivo == "percEE"].assign(variante="base")
filas = []
for (f, v), r in pd.concat([base_ee, SENS]).groupby(["familia", "variante"], sort=False):
    fila = {"Familia": FAMILIAS[f], "Variante": v, "filas": len(conjunto(f, "percEE", v)),
            "recetas": conjunto(f, "percEE", v).meta_formulation_id.nunique()}
    for esq, nombre in [("medicion", "por medición"), ("receta", "por receta")]:
        s = r[r.esquema == esq]
        fila[f"r² RF · {nombre}"] = mediana_rango(s.rf_rsq)
        fila[f"RMSE RF · {nombre}"] = mediana_rango(s.rf_rmse, 2)
    fila["r² (1−SSE/SST) · por receta"] = mediana_rango(r[r.esquema == "receta"].rf_r2sse)
    filas.append(fila)
SENS_TABLA = pd.DataFrame(filas)
SENS_TABLA.to_csv(SALIDA / "sensibilidad_ee.csv", index=False)
SENS_TABLA

## 8. Veredicto de las hipótesis

Se calcula automáticamente con los criterios registrados en `docs/parte_D_diseno.md` §5.

In [ ]:
def fila_v(hip, criterio, observado, ok, parcial=False):
    return {"Hipótesis": hip, "Criterio (fijado antes)": criterio, "Resultado observado": observado,
            "Veredicto": "✅ se acepta" if ok else ("🟡 parcial" if parcial else "❌ se rechaza")}


veredicto = []
# H1a
ok = H1_PRUEBAS["H1a"].str.startswith("✅")
veredicto.append(fila_v("H1a · el r² baja con recetas nuevas",
                        "mediana caída r² > 0 y p Holm (t corregida) < 0.05, en cada modelo",
                        f"se cumple en {ok.sum()} de 6 modelos: " + ", ".join(H1_PRUEBAS.loc[ok, "Modelo"]),
                        ok.all(), ok.any()))
# H1b
top3 = set(H1_PRUEBAS.nlargest(3, "mediana caída r²").Modelo)
esperados = {ETIQUETA[("lipidos", "pdi")], ETIQUETA[("plga", "pdi")], ETIQUETA[("lipidos", "hdiam_nm")]}
veredicto.append(fila_v("H1b · sobre todo PdI y lípidos · dH", "esos 3 modelos son las 3 caídas más grandes",
                        "3 caídas más grandes: " + ", ".join(H1_PRUEBAS.nlargest(3, "mediana caída r²").Modelo),
                        top3 == esperados, len(top3 & esperados) >= 2))
# H1c
rmse_rec = {(f, o): REP[(REP.familia == f) & (REP.objetivo == o) & (REP.esquema == "receta")].rf_rmse.median() for f, o in MODELOS}
c = [rmse_rec[m] > RUIDO[m] for m in MODELOS]
veredicto.append(fila_v("H1c · el RMSE supera el ruido", "RMSE mediano (por receta) > SD entre réplicas, en cada modelo",
                        f"se cumple en {sum(c)} de 6 modelos", all(c), any(c)))
# H1-control
okc = CONTROL_FUGA["H1-control"].str.startswith("✅")
veredicto.append(fila_v("H1-control · la fuga aislada reduce el error", "mediana RMSE(sin) − RMSE(con) > 0 y p Holm < 0.05",
                        f"se cumple en {okc.sum()} de 6 modelos", okc.all(), okc.any()))
# H2
h2 = H2_TABLA.set_index(["familia", "objetivo"])["RMSE / SD · por receta"]
a = h2[("plga", "hdiam_nm")] <= 1.5 and h2[("plga", "percEE")] <= 1.5
b = h2[("lipidos", "pdi")] > 2 and h2[("plga", "pdi")] > 2
veredicto.append(fila_v("H2 · PLGA dH/EE cerca del ruido; PdI muy por encima",
                        "RMSE/SD ≤ 1.5 en PLGA·dH y PLGA·EE; > 2 en los dos PdI",
                        f"PLGA·dH {h2[('plga', 'hdiam_nm')]:.2f}, PLGA·EE {h2[('plga', 'percEE')]:.2f}; "
                        f"lípidos·PdI {h2[('lipidos', 'pdi')]:.2f}, PLGA·PdI {h2[('plga', 'pdi')]:.2f}", a and b, a or b))
# H3
h3 = H3_TABLA.set_index(["familia", "objetivo"])
cob = h3.loc[("plga", "hdiam_nm"), "cobertura % · conformal calibrado por receta"]
veredicto.append(fila_v("H3a · cobertura cercana a la nominal en PLGA · dH", "cobertura media del conformal por receta entre 85 y 95 %",
                        f"{cob:.1f} %", 85 <= cob <= 95))
ancho = h3["ancho relativo · conformal calibrado por receta"]
otros = [("lipidos", "hdiam_nm"), ("lipidos", "pdi"), ("lipidos", "percEE"), ("plga", "pdi")]
mayores = [ancho[m] > ancho[("plga", "hdiam_nm")] for m in otros]
veredicto.append(fila_v("H3b · más anchos (rel. al ruido) en PdI y lípidos", "ancho relativo > el de PLGA · dH en esos 4 modelos",
                        f"PLGA·dH {ancho[('plga', 'hdiam_nm')]:.2f}; mayor en {sum(mayores)} de 4", all(mayores), any(mayores)))
cob_med = h3["cobertura % · conformal calibrado por medición"]
veredicto.append(fila_v("H3c · calibrar por medición sobreestima la confianza", "cobertura < 85 % en al menos 3 de 6 modelos",
                        f"< 85 % en {(cob_med < 85).sum()} de 6 (rango {cob_med.min():.1f}–{cob_med.max():.1f} %)",
                        (cob_med < 85).sum() >= 3))
VEREDICTO = pd.DataFrame(veredicto)
VEREDICTO.to_csv(SALIDA / "veredicto_hipotesis.csv", index=False)
with pd.option_context("display.max_colwidth", 120):
    display(VEREDICTO)

## 9. Interpretación de los resultados

> Corresponde a la ejecución del 17-nov-2026: 100 repeticiones por esquema, semillas de partición 0–99, `random_state = 1987`, 6 procesos y 57 min. Los criterios son los registrados en `docs/parte_D_diseno.md` **antes** de ejecutar. No se modificó ninguno.

### 9.1 Resumen del veredicto

| Hipótesis | Veredicto | En una frase |
|---|---|---|
| **H1a** · el r² baja con recetas nuevas | 🟡 Parcial (2 de 6 con la prueba *t* corregida) | El r² baja en los 6 modelos (en el 73–98 % de las repeticiones), pero la prueba *t* corregida, muy conservadora, solo lo confirma en PLGA · dH y PLGA · PdI |
| **H1b** · sobre todo PdI y lípidos · dH | ✅ Se acepta | Las 3 caídas más grandes son PLGA · PdI (−0.33), lípidos · dH (−0.12) y lípidos · PdI (−0.11) |
| **H1c** · el RMSE supera el ruido | ✅ Se acepta | En los 6 modelos, el error con recetas nuevas es mayor que la variación entre réplicas |
| **H1-control** · la fuga aislada reduce el error | ✅ Se acepta (6 de 6) | Con **las mismas filas de prueba**, no haber visto las réplicas aumenta el RMSE entre 1.3 y 1.6 veces (p de Holm < 10⁻¹¹) |
| **H2** · PLGA dH/EE cerca del ruido; PdI muy por encima | ❌ Se rechaza | Pasa **al revés** de lo esperado: PLGA · dH es el modelo más alejado del ruido (3.2 veces) y PdI el más cercano (1.5–1.7 veces) |
| **H3a** · cobertura nominal en PLGA · dH | ✅ Se acepta | El intervalo conformal calibrado por receta cubre el 92.7 % (90.5–93.5 % en los 6 modelos) |
| **H3b** · intervalos más anchos en PdI y lípidos | ❌ Se rechaza | En relación con el ruido, el intervalo más ancho es el de PLGA · dH (3.7 veces el ruido). Es coherente con H2 |
| **H3c** · calibrar por medición sobreestima la confianza | ✅ Se acepta (6 de 6) | Los intervalos "al 90 %" calibrados como valida el paper cubren solo el 72–85 % de las recetas nuevas |

### 9.2 H1: el modelo predice peor las recetas nuevas, y la causa es la fuga

- **La caída es sistemática.** Con partición por receta, el r² mediano del RF baja en los 6 modelos y el RMSE sube entre un 27 % y un 67 %.
  - Los casos más fuertes: PLGA · PdI (0.64 → 0.29), lípidos · dH (0.87 → 0.76) y lípidos · PdI (0.81 → 0.69).
  - Los modelos con r² muy alto (EE de PLGA 0.99 → 0.98; dH de PLGA 0.96 → 0.89) bajan menos en r², pero su RMSE sube igual (+55 % y +67 %).
- **Por qué H1a solo es parcial: la prueba elegida es muy conservadora.**
  - La corrección de Nadeau y Bengio multiplica la varianza por (1/J + n_prueba/n_entren.) / (1/J) ≈ 19 veces. Con eso, solo 2 de 6 caídas son significativas.
  - La prueba de Wilcoxon da p < 10⁻⁷ en los 6 modelos, pero no estaba registrada como criterio principal, así que el veredicto se mantiene en **parcial**. Ajustar el criterio después de ver los resultados sería justamente lo que las normas penalizan.
- **El experimento controlado elimina la duda.** Con filas de prueba idénticas, la única diferencia es si el modelo vio o no las otras réplicas de esas recetas. El RMSE mediano empeora entre 1.27 y 1.63 veces, y la diferencia es significativa en los 6 modelos.

  👉 **La ventaja del esquema del paper se debe a que el modelo ya vio la receta, no a que haya aprendido la física.**
- **El ajuste de hiperparámetros no cambia la conclusión (§6).**
  - En PLGA · PdI, ajustarlos con validación anidada por grupos mejora el r² solo +0.02 (p = 0.45).
  - Como era de esperar, el ajuste por grupos prefiere hojas más grandes: `min_n` de 20 o 40 en 12 de 20 particiones, frente al `min_n = 2` de la Parte B, que favorece la memorización. Aun así, no recupera lo perdido.
- **La ventaja del RF sobre la regresión lineal casi desaparece** en PLGA · dH (r² = 1 − SSE/SST: 0.871 vs 0.868) y en lípidos · EE (0.895 vs 0.899).
  - En la validación del paper (Tabla S2), el RF superaba a la regresión lineal en PLGA · dH (0.935 vs 0.882).
  - Es lo que describen Kapoor y Narayanan (2023): al corregir la fuga, los modelos complejos pierden parte de su ventaja aparente.
  - En lípidos · dH, lípidos · PdI y PLGA · PdI el RF sigue siendo claramente mejor que la regresión lineal.

### 9.3 Comparación con la línea base (Parte B) y con la Tabla 2 del paper

Las 100 repeticiones **por medición** son la "Tabla 2 con muchas semillas" (§3.3) y confirman, ahora con el RF real, lo que la Parte C había aproximado con el memorizador:

- **5 de los 6 valores del paper caen dentro de la variación por partición** (percentiles 14–77).
- **El valor del paper en lípidos · EE (r² = 0.472) queda por debajo de las 100 repeticiones.** No es efecto del azar: el RF da entre 0.876 y 0.975 en el 90 % de las particiones.
- **Los valores del código R del autor** están entre los percentiles 14 y 89.
- **Los de la Parte B (Python)** están entre los percentiles 23 y 85, salvo lípidos · EE (0.827), que también queda en el extremo bajo.
- **La variación por partición es grande:** el rango del 90 % central mide entre 0.02 (PLGA · EE) y 0.25 (PLGA · PdI) de r². Con partición por receta es todavía mayor (hasta 0.54 en lípidos · dH).

  👉 Un solo número de r² de prueba, como el de la Tabla 2, no basta para describir el rendimiento del modelo.

### 9.4 H2 y H3: el error comparado con el ruido, y la incertidumbre

- **H2 falla porque el ruido es muy distinto entre variables, no porque el modelo sea malo en todas.** El cociente RMSE / ruido es pequeño cuando la medición es ruidosa y grande cuando la medición es precisa:

  | Modelo | Ruido entre réplicas | Techo de r² (Parte C) | Error con recetas nuevas | Cociente error / ruido |
  |---|---|---|---|---|
  | PLGA · dH | muy bajo (SD mediana de 1.6 nm) | 0.993 | 14.7 nm | **3.2** |
  | Lípidos · PdI | alto | 0.90 | — | **1.5** |
  | PLGA · PdI | alto | 0.83 | — | **1.7** |

- **El ancho de los intervalos (H3b) sigue el mismo patrón.** Por eso el intervalo de PLGA · dH es el más ancho en relación con el ruido.
- **Los intervalos calibrados por receta funcionan** (cobertura del 90.5–93.5 %), pero son anchos:
  - ±28 nm para el dH de PLGA (ancho mediano de 55 nm);
  - ±45 nm para el dH de lípidos;
  - ±25 % de EE en lípidos.
- **Los intervalos que se obtienen si se confía en la validación del paper son engañosos:**
  - el conformal calibrado por medición dice "90 %", pero cubre entre el 72 y el 85 % de las recetas nuevas;
  - el Quantile Random Forest cubre entre el 72 y el 91 %, porque sus hojas pequeñas (`min_n = 2`) reflejan sobre todo la dispersión entre réplicas de recetas ya vistas.

### 9.5 Sensibilidad de la EE: el r² de lípidos se debe a la regla "con o sin fármaco"

| Familia | Datos | r² (correlación²) con partición por receta | r² = 1 − SSE/SST con partición por receta |
|---|---|---|---|
| Lípidos | completos | 0.904 | 0.895 |
| Lípidos | solo liposomas con curcumina | **0.110** | **−0.005** |
| PLGA | completos | 0.980 | 0.978 |
| PLGA | solo con fármaco | **0.930** | 0.914 |
| Lípidos | EE recortada a 100 % | 0.908 | 0.904 |

- **Lípidos:** con todos los datos, 2 de cada 3 filas tienen EE = 0 porque no llevan fármaco. Al dejar solo los liposomas con curcumina, el modelo **no predice mejor que la media** cuando la receta es nueva.
- **PLGA:** el modelo conserva casi todo su rendimiento.
- **Recortar las EE mayores de 100 %** casi no cambia nada.

## 10. Discusión física y química

**Contexto.** La formación de nanopartículas por microfluídica (nanoprecipitación de PLGA y autoensamblaje de lípidos) se explica como una **competencia entre dos tiempos** (Karnik *et al.*, 2008; Jahn *et al.*, 2010; Zhigaltsev *et al.*, 2012):

- **el tiempo de mezcla**, que acorta el TFR (caudal total);
- **el tiempo de nucleación, crecimiento y agregación** del material, que depende de la sobresaturación. Esta a su vez depende de la concentración de polímero o lípido y de la fracción final de solvente, 1/(1 + FRR).

Los resultados de la Parte D se leen así:

1. **El tamaño de PLGA es una propiedad suave y casi monótona de la receta. El modelo la capta, pero con menos precisión de la que sugiere el paper.**
   - Con recetas nuevas, la regresión lineal iguala al RF (r² ≈ 0.87). La relación entre tamaño y TFR, FRR y concentración se parece más a una ley de escala sencilla (más polímero → partículas más grandes; mezcla más rápida y más dilución → más pequeñas) que a una función compleja.
   - La ventaja del RF en la validación del paper venía de "recordar" el tamaño exacto de cada receta. Eso incluye desviaciones propias de cada receta que la composición no explica.
   - **¿Por qué el error (14.7 nm) es 3 veces mayor que el ruido de la DLS?**
     - El diseño experimental tiene solo 3 niveles de TFR (3, 6 y 13) y 3 de FRR (3, 4 y 5). Una receta nueva suele diferir de sus vecinas en más de un factor, y el RF predice "por escalones".
     - Además, puede haber variables no registradas que cambian de una receta a otra: lote de polímero, día u operador (`meta_commit_id`). La medición es precisa (techo de r² 0.993), pero **los predictores no contienen toda la información** que fija el tamaño.
   - Esta explicación no se contrastó en este trabajo.
2. **La polidispersidad (PdI) está dominada por el ruido de medición.**
   - El PdI de la DLS sale del análisis de cumulantes. Es poco preciso en valores bajos y en muestras con agregados, y depende de la preparación de la muestra.
   - El ruido entre réplicas deja un techo de r² de 0.83–0.90, y entre recetas hay poca variación real que explicar.
   - Por eso:
     - el error del modelo queda cerca del ruido en valor absoluto (1.5–1.7 veces);
     - pero el r² se desploma con recetas nuevas, especialmente en PLGA (0.29);
     - con partición por medición, el modelo "acertaba" al recordar el PdI de cada receta, que sí se repite en parte entre réplicas (por ejemplo, por agregados característicos de una mezcla).
   - **Implicación práctica:** el PdI no se debería optimizar por diseño inverso con este modelo sin replicar mucho más cada receta.
3. **En PLGA, la encapsulación sí depende de la química. En los liposomas de este conjunto, el modelo no la predice.**
   - **En PLGA**, la curcumina (muy hidrofóbica) coprecipita con el polímero. La EE queda determinada por la proporción fármaco:polímero y por la sobresaturación, y el modelo conserva un r² de 0.93 incluso solo con recetas con fármaco.
   - **En los liposomas**, la curcumina se aloja en la bicapa y puede competir por espacio con el colesterol. Su EE depende de la composición lipídica y de la proporción lípido:fármaco, pero con 27 recetas y una medición ruidosa (9 valores > 100 % sugieren problemas en la separación o en la cuantificación por UV-Vis), el modelo **no supera a la media** con recetas nuevas.
   - El r² alto de la EE de lípidos en el paper refleja sobre todo una regla trivial: **sin fármaco, EE = 0**.
4. **Consecuencia para el diseño inverso**, que es el objetivo del paper.
   - Las recetas que se proponen con el modelo son, por definición, nuevas. Su incertidumbre real es la de los intervalos calibrados por receta: ±28 nm en el tamaño de PLGA y ±45 nm en el de lípidos.
   - Sirven para orientar la búsqueda (distinguir partículas de 80 nm y de 200 nm), pero **no para garantizar un tamaño objetivo con la precisión que sugieren el r² de 0.96 y el RMSE de 7.5 nm de la Tabla 2**.

## 11. Limitaciones

- **La prueba *t* corregida empareja las repeticiones por número de semilla**, aunque las filas de prueba de los dos esquemas no coinciden. Es conservadora. El experimento controlado (§3.4) sí está emparejado exactamente.
- **Los conjuntos de prueba son pequeños:** 12–20 recetas por repetición. Por eso cada repetición varía mucho, y se resumen con medianas y percentiles.
- **Hiperparámetros fijos.** El control con ajuste anidado se hizo solo en PLGA · PdI, con 20 particiones.
- **Cobertura media, no por receta.** El conformal garantiza la cobertura **en promedio**, no para cada región del espacio de recetas. Además, supone que las recetas nuevas se parecen a las del conjunto (el diseño es una rejilla de niveles). La extrapolación fuera de esos niveles no se evaluó.
- **Solo se usó el Random Forest** (más las referencias lineal y memorizador). Los 13 algoritmos de las Tablas S1/S2 no se compararon con partición por receta.
- **No se modelaron efectos de lote** (operador, día) ni la validación experimental del autor con recetas nuevas (`recursos/data/data_genAI_v_mfNP.csv`). Esa validación sería la prueba externa natural de estas conclusiones.

## 12. Conclusiones

1. **La validación del paper sobreestima la capacidad del modelo para predecir recetas nuevas.**
   - Con partición por receta, el r² baja en los 6 modelos (hasta −0.33 en PLGA · PdI) y el error sube entre un 27 y un 67 %.
   - El experimento controlado muestra que la causa es haber visto las réplicas: el error aumenta entre 1.3 y 1.6 veces con las mismas filas de prueba.
2. **El tamaño y la EE de PLGA se siguen prediciendo bien con recetas nuevas** (r² 0.89 y 0.98).
   - Su relación con la receta es suave: la regresión lineal rinde casi igual que el RF en el tamaño.
   - Aun así, el error del tamaño es 3 veces el ruido de medición.
3. **El PdI y la EE de los liposomas con fármaco no se pueden predecir de forma útil para recetas nuevas** con estos datos:
   - el PdI, por el ruido de medición;
   - la EE de los liposomas, porque sin la regla "con o sin fármaco" no supera a la media.
4. **Para usar el modelo en diseño inverso hay que acompañar cada predicción con un intervalo calibrado por receta.**
   - Los intervalos al 90 % calibrados así cubren el 90.5–93.5 % de las recetas nuevas.
   - Los que se obtienen con la validación del paper cubren solo el 72–85 %.

**Referencias nuevas de esta sección** (además de las de la Parte C y de `docs/parte_D_diseno.md`):
- Karnik, R. *et al.* Microfluidic platform for controlled synthesis of polymeric nanoparticles. *Nano Lett.* 2008, 8, 2906–2912.
- Jahn, A. *et al.* Microfluidic mixing and the formation of nanoscale lipid vesicles. *ACS Nano* 2010, 4, 2077–2087.
- Zhigaltsev, I. V. *et al.* Bottom-up design and synthesis of limit size lipid nanoparticle systems with aqueous and triglyceride cores using millisecond microfluidic mixing. *Langmuir* 2012, 28, 3633–3640.
- Kapoor, S.; Narayanan, A. Leakage and the reproducibility crisis in machine-learning-based science. *Patterns* 2023, 4, 100804.